# ValidMind + Red Hat OpenShift AI Quickstart

Use this notebook to install and run the ValidMind Library inside a **Red Hat OpenShift AI workbench**, load data from an S3-compatible object store linked to your workbench through a **connection**, train a simple classification model, and send the results to the ValidMind Platform.

In this notebook, you will:

- Install and initialize the ValidMind Library
- Load data from an S3-compatible object store using your workbench connection
- Train a simple classification model
- Run ValidMind tests and send the results to the ValidMind Platform

## Before you begin

You will need:

1. A running **OpenShift AI workbench** (JupyterLab), created from a data science project. The **Standard Data Science** or **PyTorch/TensorFlow** notebook image works well.
2. A ValidMind account with a registered model.
3. Your ValidMind API credentials (API key, API secret, model identifier).

To get your credentials: log in to ValidMind → **Model Inventory** → select your model → **Getting Started** → **Copy snippet to clipboard**.

> **Note:** If you don't have an S3 connection with training data yet, this notebook includes a synthetic-data fallback so you can still run through the full workflow.

### How this maps to your OpenShift AI project

| OpenShift AI concept | How this notebook uses it |
|----------------------|---------------------------|
| **Workbench** (JupyterLab pod) | The environment you're running this notebook in. |
| **Connection** (S3-compatible) | Injects `AWS_S3_ENDPOINT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_BUCKET`, and `AWS_DEFAULT_REGION` as environment variables. Step 4 reads your dataset from here. |
| **Cluster storage** (PVC) | Persists this notebook and any downloaded data across workbench restarts. |
| **Secret / environment variables** | Where your ValidMind API credentials are read from in Step 3. |

## Step 1 — Install the ValidMind Library

Run this cell first. In a workbench, packages you install with `pip` land in the container's ephemeral environment (or your PVC, depending on how the image is configured), so you'll need to reinstall after the workbench restarts unless you bake the dependency into a custom image. The default package index configured in some OpenShift AI workbench images does not carry `validmind` (yet), so the install below points explicitly at PyPI.

In [ ]:
%pip install -q --index-url https://pypi.org/simple validmind "plotly<6" "kaleido==0.2.1"

After the install finishes, **restart the kernel** so the new package is picked up: in JupyterLab, go to **Kernel → Restart Kernel…** (or press `0` `0`). There is no `dbutils.library.restartPython()` in OpenShift AI — kernel restarts are driven from the JupyterLab menu.

## Step 2 — Verify installation

Confirm that the ValidMind Library installed successfully and check the version available in your workbench:

In [ ]:
import importlib.metadata

version = importlib.metadata.version('validmind')
print(f'ValidMind Library version: {version}')
print('Installation successful!')

## Step 3 — Initialize the ValidMind Library

Initialize the ValidMind Library so test results are uploaded to the correct model in the ValidMind Platform.

You can supply your credentials in any of three ways:

- **ValidMind connection** (recommended): if your cluster admin installed this quickstart's *ValidMind connection type*, create a connection in your data science project (**Connections → Create connection → ValidMind**) and attach it to this workbench. The workbench then has `VM_API_HOST`, `VM_API_KEY`, `VM_API_SECRET`, and `VM_API_MODEL` set as environment variables — the deployment created by `make install` does this for you — and `vm.init()` needs no arguments.
- **Environment variables / OpenShift Secret**: add the same four variables when creating or editing the workbench (**Environment variables → Add variable**, either as key/value pairs or from a Secret). This keeps secrets out of the notebook.
- **Edit the next cell directly**: uncomment the `os.environ[...]` lines and paste your credentials.

To get your credentials:

1. In ValidMind, go to **Model Inventory** and select your model.
2. Open **Getting Started** and click **Copy snippet to clipboard**.
3. Set the corresponding environment variables, or paste the values into the next cell.

In [ ]:
import os
import validmind as vm

# ---------------------------------------------------------------------------
# The ValidMind Library reads VM_API_HOST, VM_API_KEY, VM_API_SECRET, and
# VM_API_MODEL from the environment. If a ValidMind connection is attached to
# this workbench (or `make install` deployed it), they are already set and
# vm.init() needs no arguments.
#
# Running elsewhere? Uncomment and fill in the values from your model's
# "Getting Started" page in the ValidMind Platform:
# ---------------------------------------------------------------------------
# os.environ["VM_API_HOST"]   = "https://app.prod.validmind.ai/api/v1/tracking"
# os.environ["VM_API_KEY"]    = "<YOUR_API_KEY>"
# os.environ["VM_API_SECRET"] = "<YOUR_API_SECRET>"
# os.environ["VM_API_MODEL"]  = "<YOUR_MODEL_CUID>"

missing = [v for v in ("VM_API_KEY", "VM_API_SECRET", "VM_API_MODEL") if not os.environ.get(v)]
if missing:
    raise RuntimeError(
        f"Missing ValidMind credentials: {', '.join(missing)}. Attach a ValidMind "
        "connection to this workbench, or uncomment the os.environ lines above."
    )

vm.init()

## Step 4 — Load data from your S3-compatible object store

Load the data for this notebook from the S3-compatible object store that your workbench **connection** points to (for example, MinIO, OpenShift Data Foundation, or AWS S3). When you attach a connection to a workbench, OpenShift AI injects these environment variables into the pod:

- `AWS_S3_ENDPOINT` — the object store endpoint URL
- `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` — your credentials
- `AWS_S3_BUCKET` — the default bucket
- `AWS_DEFAULT_REGION` — the region (optional for MinIO)

The next cell builds a `boto3` client from those variables and reads a CSV object into a pandas DataFrame. Set `DATA_OBJECT_KEY` to the path of your training data within the bucket.

If no S3 connection is attached to the workbench, the next cell automatically falls back to a generated synthetic dataset so you can still run the full workflow end-to-end. Set `USE_SYNTHETIC_FALLBACK` yourself to force either behavior.

In [ ]:
import os
import io
import pandas as pd

# Auto-detect: use the S3 connection when one is attached to the workbench,
# otherwise fall back to a generated synthetic dataset. Override by setting
# this to True or False explicitly.
USE_SYNTHETIC_FALLBACK = not os.environ.get("AWS_S3_ENDPOINT")

# Path to your CSV object within the bucket, and the target column name
DATA_OBJECT_KEY = "data/training_data.csv"  # <-- update to your object key
TARGET_COLUMN   = "target"                  # <-- update to your target column

# ---------------------------------------------------------------------------
# Load from the S3-compatible object store using the workbench connection
# ---------------------------------------------------------------------------
if not USE_SYNTHETIC_FALLBACK:
    import boto3

    endpoint_url = os.environ.get("AWS_S3_ENDPOINT")
    access_key   = os.environ.get("AWS_ACCESS_KEY_ID")
    secret_key   = os.environ.get("AWS_SECRET_ACCESS_KEY")
    region       = os.environ.get("AWS_DEFAULT_REGION")
    bucket       = os.environ.get("AWS_S3_BUCKET")

    if not all([endpoint_url, access_key, secret_key, bucket]):
        raise RuntimeError(
            "S3 connection variables are not set. Attach a connection to this "
            "workbench (Workbench → Connections), or set "
            "USE_SYNTHETIC_FALLBACK = True above to continue with generated data."
        )

    session = boto3.session.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
    )
    s3 = session.client(
        "s3",
        endpoint_url=endpoint_url,
        region_name=region,
    )

    obj = s3.get_object(Bucket=bucket, Key=DATA_OBJECT_KEY)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))

    if TARGET_COLUMN not in df.columns:
        raise ValueError(
            f"Column '{TARGET_COLUMN}' not found in {DATA_OBJECT_KEY}. "
            f"Available columns: {list(df.columns)}. "
            "Update TARGET_COLUMN above to match your data's target column."
        )

    print(f"Loaded {len(df):,} rows, {len(df.columns)} columns from s3://{bucket}/{DATA_OBJECT_KEY}")
    print(f"Target distribution: {df[TARGET_COLUMN].value_counts().to_dict()}")
    df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Synthetic data fallback — runs when USE_SYNTHETIC_FALLBACK = True
# Uses the Bank Customer Churn dataset pattern from ValidMind examples
# ---------------------------------------------------------------------------
if USE_SYNTHETIC_FALLBACK:
    import numpy as np
    from sklearn.datasets import make_classification

    np.random.seed(42)
    X, y = make_classification(
        n_samples=1000,
        n_features=10,
        n_informative=6,
        n_redundant=2,
        random_state=42,
    )
    feature_names = [
        "credit_score", "age", "tenure", "balance",
        "num_products", "has_credit_card", "is_active_member",
        "estimated_salary", "geography_encoded", "gender_encoded",
    ]
    df = pd.DataFrame(X, columns=feature_names)
    df["target"] = y
    TARGET_COLUMN = "target"

    print(f"Using synthetic dataset: {len(df):,} rows, {len(df.columns)} columns")
    print(f"Target distribution: {df[TARGET_COLUMN].value_counts().to_dict()}")
    df.head()

## Step 5 — Prepare train/test split

Split the dataset into a training set and a test set so you can train the model on one slice of the data and evaluate how it generalizes on data it hasn't seen:

In [ ]:
from sklearn.model_selection import train_test_split

feature_columns = [c for c in df.columns if c != TARGET_COLUMN]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f'Train set: {len(train_df):,} rows')
print(f'Test set:  {len(test_df):,} rows')
print(f'Features:  {feature_columns}')

## Step 6 — Train a simple model

Train a gradient boosting classifier on the training set. This is a small, fast model that's well-suited to a quickstart. The goal here is to produce something documentable end-to-end, not to tune for accuracy.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(train_df[feature_columns], train_df[TARGET_COLUMN])

train_accuracy = model.score(train_df[feature_columns], train_df[TARGET_COLUMN])
test_accuracy  = model.score(test_df[feature_columns],  test_df[TARGET_COLUMN])

print(f'Train accuracy: {train_accuracy:.4f}')
print(f'Test accuracy:  {test_accuracy:.4f}')

## Step 7 — Register datasets and model with ValidMind

Before you can run tests, ValidMind needs to know about your datasets and your model. Wrap the training and test DataFrames with [`vm.init_dataset()`](https://docs.validmind.ai/validmind/validmind.html#init_dataset) and the trained classifier with [`vm.init_model()`](https://docs.validmind.ai/validmind/validmind.html#init_model). Each call returns a ValidMind object that the test functions accept as input.

The `input_id` you pass identifies each input when results are sent to the ValidMind Platform.

In [ ]:
vm_train_ds = vm.init_dataset(
    dataset=train_df,
    input_id="train_dataset",
    target_column=TARGET_COLUMN,
)

vm_test_ds = vm.init_dataset(
    dataset=test_df,
    input_id="test_dataset",
    target_column=TARGET_COLUMN,
)

vm_model = vm.init_model(
    model=model,
    input_id="gradient_boosting_model",
)

print('Datasets and model registered with ValidMind.')

## Step 8 — Assign predictions to datasets

Many tests compare predicted values against actual values, so ValidMind needs the model's predictions attached to each dataset. The [`assign_predictions()` method](https://docs.validmind.ai/validmind/validmind/vm_models.html#assign_predictions) computes predictions from your model and links them to the dataset object, once for the training set and once for the test set:

In [ ]:
vm_train_ds.assign_predictions(model=vm_model)
vm_test_ds.assign_predictions(model=vm_model)

print('Predictions assigned.')

## Step 9 — Run individual tests

Run a few individual tests against your registered datasets and model to get familiar with how ValidMind tests work before running the full suite. Each [`vm.tests.run_test()`](https://docs.validmind.ai/validmind/validmind/tests.html#run_test) call executes one test, renders the result inline in this notebook, and `result.log()` sends the result to the ValidMind Platform:

In [ ]:
# Dataset statistics — validates data documentation capability
result = vm.tests.run_test(
    "validmind.data_validation.DatasetDescription",
    inputs={"dataset": vm_train_ds},
)
result.log()

In [ ]:
# Class imbalance check
result = vm.tests.run_test(
    "validmind.data_validation.ClassImbalance",
    inputs={"dataset": vm_train_ds},
)
result.log()

In [ ]:
# Confusion matrix — validates model performance visualization
result = vm.tests.run_test(
    "validmind.model_validation.sklearn.ConfusionMatrix",
    inputs={"dataset": vm_test_ds, "model": vm_model},
)
result.log()

In [ ]:
# ROC curve
result = vm.tests.run_test(
    "validmind.model_validation.sklearn.ROCCurve",
    inputs={"dataset": vm_test_ds, "model": vm_model},
)
result.log()

In [ ]:
# Feature importance
result = vm.tests.run_test(
    "validmind.model_validation.sklearn.FeatureImportance",
    inputs={"dataset": vm_train_ds, "model": vm_model},
)
result.log()

## Step 10 — Run the full test suite

Run the complete classifier documentation suite. This single call executes every test in the suite and sends all results to the ValidMind Platform, where they populate the corresponding sections of your model documentation:

In [ ]:
test_suite_result = vm.run_test_suite(
    "classifier_full_suite",
    inputs={
        "dataset": vm_test_ds,
        "datasets": [vm_train_ds, vm_test_ds],
        "model": vm_model,
        "train_dataset": vm_train_ds,
        "test_dataset": vm_test_ds,
    },
)
print('Full test suite completed and results sent to ValidMind Platform.')

## Step 11 — Verify results on the platform

To see the results of this notebook in the ValidMind Platform:

1. Go to the [ValidMind Platform](https://app.prod.validmind.ai) (or your ValidMind instance).
2. Navigate to **Model Inventory** and select your model.
3. Open the **Documentation** tab.
4. Confirm that the test results from this notebook appear in the relevant sections.

After a successful run, you should see the following results in your model's documentation:

- Dataset Description table
- Class Imbalance chart
- Confusion Matrix
- ROC Curve
- Feature Importance chart
- Full classifier suite results

---

## Troubleshooting

If you run into any of the issues below, the table lists the likely fix:

| Issue | Fix |
|-------|-----|
| `ModuleNotFoundError` after install | Restart the kernel (**Kernel → Restart Kernel…**) so the newly installed package is picked up. |
| `ConnectionError` on `vm.init()` | Your cluster may restrict egress. Check the project's network policies / egress firewall, or ask your cluster admin to allow outbound traffic to the ValidMind API host. |
| `401 Unauthorized` on `vm.init()` | The API key or secret is incorrect. Copy your credentials again from the ValidMind Platform, or check the environment variables on your workbench. |
| `numpy` version conflict | Pin a compatible version with `%pip install -q validmind "numpy>=1.23,<2.0.0"`, then restart the kernel. |
| S3 connection variables not set | No connection is attached. Edit the workbench and attach a connection (**Connections**), or add the `AWS_*` variables, then restart the workbench. |
| `NoSuchKey` / `NoSuchBucket` from boto3 | `DATA_OBJECT_KEY` or the bucket is wrong. Confirm the object exists with `s3.list_objects_v2(Bucket=bucket)`. |
| `SSL`/endpoint errors against MinIO | Some MinIO setups need the endpoint without a scheme, or `verify=False` on the client for self-signed certs. |
| Wrong columns or target not found | Update `TARGET_COLUMN` (and `DATA_OBJECT_KEY`) in Step 4 to match your data. |
| Want to try the notebook without a connection | Set `USE_SYNTHETIC_FALLBACK = True` in Step 4 to use generated data. |
| `Failed to run test ...SHAPGlobalImportance` during the full suite | Benign: the optional `shap` package is not installed. Add `shap` to the Step 1 install cell if you want SHAP plots. |
| `Failed to run test ...ModelsPerformanceComparison` during the full suite | Benign: that suite test compares multiple models and this quickstart trains one. |
| Work lost after workbench restart | Save notebooks and data to your **cluster storage (PVC)** mount so they persist across restarts. |

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>